In [ ]:
import os
import json
import sqlite3
import numpy as np
import pandas as pd

DB_PATH     = 'logs.db'
OUT_DIR     = '../data/graph_prep'
os.makedirs(OUT_DIR, exist_ok=True)

BUCKET_SIZE     = 3600   # seconds per bucket (1 hour)
ROLLING_WINDOW  = 24     # 24-bucket (24h) rolling window for z-scores

# Zeek conn_state values that indicate a failed/incomplete connection
FAILED_STATES = {'S0', 'REJ', 'RSTOS0', 'RSTO', 'SH', 'OTH', 'RSTRH', 'SHR'}

In [ ]:
# --- Load raw logs from SQLite ---
# Column names like id.orig_h need quoting in SQL

with sqlite3.connect(DB_PATH) as db:
    conn_df = pd.read_sql_query("""
        SELECT ts,
               "id.orig_h" AS src_ip,
               "id.orig_p" AS src_port,
               "id.resp_h" AS dst_ip,
               "id.resp_p" AS dst_port,
               proto, duration,
               orig_bytes, resp_bytes,
               orig_pkts, resp_pkts,
               orig_ip_bytes, resp_ip_bytes,
               conn_state, missed_bytes, service
        FROM all_conn
    """, db)

    dns_df = pd.read_sql_query("""
        SELECT ts,
               "id.orig_h" AS src_ip,
               "id.resp_h" AS resolver_ip,
               query, rcode_name, qtype_name
        FROM all_dns
    """, db)

    http_df = pd.read_sql_query("""
        SELECT ts,
               "id.orig_h" AS src_ip,
               "id.resp_h" AS dst_ip,
               method, status_code,
               request_body_len, response_body_len,
               user_agent, host
        FROM all_http
    """, db)

    files_df = pd.read_sql_query("""
        SELECT ts, tx_hosts, rx_hosts,
               mime_type, seen_bytes, is_orig
        FROM all_files
    """, db)

    weird_df = pd.read_sql_query("""
        SELECT ts,
               "id.orig_h" AS src_ip,
               "id.resp_h" AS dst_ip,
               name
        FROM all_weird
        WHERE "id.orig_h" IS NOT NULL AND "id.resp_h" IS NOT NULL
    """, db)

    tunnel_df = pd.read_sql_query("""
        SELECT ts,
               "id.orig_h" AS src_ip,
               "id.resp_h" AS dst_ip,
               tunnel_type
        FROM all_tunnel
    """, db)

print(f"conn={len(conn_df):,}  dns={len(dns_df):,}  http={len(http_df):,}  "
      f"files={len(files_df):,}  weird={len(weird_df):,}  tunnel={len(tunnel_df):,}")

In [ ]:
# --- Assign hourly bucket IDs from raw timestamps ---
# Bucket 0 = first hour of data; bucket N = N hours later

MIN_TS = conn_df['ts'].min()
MAX_TS = conn_df['ts'].max()

def assign_bucket(df, min_ts=MIN_TS, size=BUCKET_SIZE):
    df = df.copy()
    df['bucket_id'] = ((df['ts'] - min_ts) / size).astype(int)
    return df

conn_df   = assign_bucket(conn_df)
dns_df    = assign_bucket(dns_df)
http_df   = assign_bucket(http_df)
files_df  = assign_bucket(files_df)
weird_df  = assign_bucket(weird_df)
tunnel_df = assign_bucket(tunnel_df)

n_buckets = conn_df['bucket_id'].nunique()
span_days = (MAX_TS - MIN_TS) / 86400
print(f"Time span: {span_days:.1f} days")
print(f"Hourly buckets with activity: {n_buckets}  (max possible: {int(span_days * 24)})")
print(f"Avg connections/bucket: {len(conn_df)/n_buckets:.0f}")

In [ ]:
# --- Classify IPs ---
# Used to label nodes and optionally filter noise (multicast, special) from edge list

def classify_ip(ip):
    if pd.isna(ip):
        return 'unknown'
    ip = str(ip).lower()
    if ip in ('0.0.0.0', '255.255.255.255', '::', '::1', '-'):
        return 'special'
    if ip.startswith(('224.', '239.')) or ip.startswith(('ff0', 'ff1', 'ff2', 'ff3')):
        return 'multicast'
    if ip.startswith('169.254.') or ip.startswith('fe80:'):
        return 'link_local'
    if (ip.startswith('10.') or ip.startswith('192.168.') or
            any(ip.startswith(f'172.{i}.') for i in range(16, 32))):
        return 'private'
    return 'external'

all_ips = pd.Series(
    pd.concat([conn_df['src_ip'], conn_df['dst_ip']]).dropna().unique(),
    name='ip'
).to_frame()
all_ips['ip_class'] = all_ips['ip'].map(classify_ip)

print("IP class distribution across all connections:")
print(all_ips['ip_class'].value_counts())

# Remove multicast and special-address connections — not meaningful graph nodes
EXCLUDE_CLASSES = {'multicast', 'special'}
ip_class_map = all_ips.set_index('ip')['ip_class']

mask = (
    ~conn_df['src_ip'].map(ip_class_map).isin(EXCLUDE_CLASSES) &
    ~conn_df['dst_ip'].map(ip_class_map).isin(EXCLUDE_CLASSES)
)
conn_df = conn_df[mask].copy()
print(f"\nConnections after filtering: {len(conn_df):,}")

In [ ]:
# --- Build edge table from conn log ---
# One row per (bucket_id, src_ip, dst_ip) with aggregated connection features

conn_df['is_failed']   = conn_df['conn_state'].isin(FAILED_STATES).astype(int)
conn_df['orig_bytes']  = conn_df['orig_bytes'].fillna(0)
conn_df['resp_bytes']  = conn_df['resp_bytes'].fillna(0)
conn_df['duration']    = conn_df['duration'].fillna(0)
conn_df['missed_bytes']= conn_df['missed_bytes'].fillna(0)
conn_df['total_bytes'] = conn_df['orig_bytes'] + conn_df['resp_bytes']

edges = conn_df.groupby(['bucket_id', 'src_ip', 'dst_ip']).agg(
    conn_count        = ('ts',             'count'),
    orig_bytes        = ('orig_bytes',      'sum'),
    resp_bytes        = ('resp_bytes',      'sum'),
    total_bytes       = ('total_bytes',     'sum'),
    avg_duration      = ('duration',        'mean'),
    max_duration      = ('duration',        'max'),
    failed_count      = ('is_failed',       'sum'),
    missed_bytes      = ('missed_bytes',    'sum'),
    total_pkts_sent   = ('orig_pkts',       'sum'),
    total_pkts_recv   = ('resp_pkts',       'sum'),
    unique_dst_ports  = ('dst_port',        'nunique'),
    unique_src_ports  = ('src_port',        'nunique'),
    tcp_conns         = ('proto',           lambda x: (x == 'tcp').sum()),
    udp_conns         = ('proto',           lambda x: (x == 'udp').sum()),
    icmp_conns        = ('proto',           lambda x: (x == 'icmp').sum()),
).reset_index()

# Derived edge ratios
edges['failed_ratio']      = edges['failed_count']  / edges['conn_count']
edges['tcp_ratio']         = edges['tcp_conns']      / edges['conn_count']
edges['traffic_asymmetry'] = (edges['orig_bytes'] - edges['resp_bytes']) / edges['total_bytes'].replace(0, 1)
edges['bytes_per_conn']    = edges['total_bytes']    / edges['conn_count']
edges['missed_ratio']      = edges['missed_bytes']   / edges['total_bytes'].replace(0, 1)

print(f"Edge table: {len(edges):,} rows  ×  {len(edges.columns)} columns")
edges.head()

In [ ]:
# --- DNS enrichment ---
# Aggregated per (src_ip, bucket_id); joined onto nodes later, not edges

dns_df['is_nxdomain'] = (dns_df['rcode_name'] == 'NXDOMAIN').astype(int)

dns_node = dns_df.groupby(['bucket_id', 'src_ip']).agg(
    dns_queries    = ('query',        'count'),
    unique_domains = ('query',        'nunique'),
    nxdomain_count = ('is_nxdomain',  'sum'),
    unique_qtypes  = ('qtype_name',   'nunique'),
).reset_index().rename(columns={'src_ip': 'node_ip'})

dns_node['nxdomain_ratio'] = dns_node['nxdomain_count'] / dns_node['dns_queries']

print(f"DNS node records: {len(dns_node):,}")

In [ ]:
# --- HTTP enrichment ---
# Aggregated per (src_ip, dst_ip, bucket_id) and merged onto edge table

http_df['status_code'] = pd.to_numeric(http_df['status_code'], errors='coerce')
http_df['is_error']    = (http_df['status_code'] >= 400).astype(int)
http_df['is_post']     = (http_df['method'] == 'POST').astype(int)
http_df['request_body_len']  = http_df['request_body_len'].fillna(0)
http_df['response_body_len'] = http_df['response_body_len'].fillna(0)

http_edge = http_df.groupby(['bucket_id', 'src_ip', 'dst_ip']).agg(
    http_requests      = ('method',              'count'),
    http_post_count    = ('is_post',             'sum'),
    http_error_count   = ('is_error',            'sum'),
    http_bytes_sent    = ('request_body_len',    'sum'),
    http_bytes_recv    = ('response_body_len',   'sum'),
    unique_user_agents = ('user_agent',          'nunique'),
    unique_hosts       = ('host',                'nunique'),
).reset_index()

http_edge['http_post_ratio']  = http_edge['http_post_count']  / http_edge['http_requests']
http_edge['http_error_ratio'] = http_edge['http_error_count'] / http_edge['http_requests']

http_cols = [c for c in http_edge.columns if c not in ('bucket_id', 'src_ip', 'dst_ip')]
edges = edges.merge(http_edge, on=['bucket_id', 'src_ip', 'dst_ip'], how='left')
edges[http_cols] = edges[http_cols].fillna(0)

print(f"Edges after HTTP merge: {len(edges):,}")

In [ ]:
# --- File transfer enrichment ---
# tx_hosts / rx_hosts are stored as JSON arrays; parse to extract first IP

def first_host(val):
    if pd.isna(val) or val in ('', '[]'):
        return None
    try:
        parsed = json.loads(val)
        return parsed[0] if parsed else None
    except (json.JSONDecodeError, TypeError):
        return str(val).strip()

files_df['src_ip']    = files_df['tx_hosts'].apply(first_host)
files_df['dst_ip']    = files_df['rx_hosts'].apply(first_host)
files_df = files_df.dropna(subset=['src_ip', 'dst_ip'])

files_df['seen_bytes']     = files_df['seen_bytes'].fillna(0)
files_df['is_executable']  = files_df['mime_type'].str.contains(
    'executable|x-dosexec|x-msdownload', case=False, na=False
).astype(int)

files_edge = files_df.groupby(['bucket_id', 'src_ip', 'dst_ip']).agg(
    file_count        = ('mime_type',     'count'),
    file_bytes        = ('seen_bytes',    'sum'),
    unique_mime_types = ('mime_type',     'nunique'),
    executable_files  = ('is_executable', 'sum'),
).reset_index()

files_cols = [c for c in files_edge.columns if c not in ('bucket_id', 'src_ip', 'dst_ip')]
edges = edges.merge(files_edge, on=['bucket_id', 'src_ip', 'dst_ip'], how='left')
edges[files_cols] = edges[files_cols].fillna(0)

print(f"Edges after files merge: {len(edges):,}")

In [ ]:
# --- Weird and tunnel enrichment ---

weird_edge = weird_df.groupby(['bucket_id', 'src_ip', 'dst_ip']).agg(
    weird_count        = ('name', 'count'),
    unique_weird_types = ('name', 'nunique'),
).reset_index()

tunnel_edge = tunnel_df.groupby(['bucket_id', 'src_ip', 'dst_ip']).agg(
    tunnel_count        = ('tunnel_type', 'count'),
    unique_tunnel_types = ('tunnel_type', 'nunique'),
).reset_index()

for enrich_df, enrich_name in [(weird_edge, 'weird'), (tunnel_edge, 'tunnel')]:
    enrich_cols = [c for c in enrich_df.columns if c not in ('bucket_id', 'src_ip', 'dst_ip')]
    edges = edges.merge(enrich_df, on=['bucket_id', 'src_ip', 'dst_ip'], how='left')
    edges[enrich_cols] = edges[enrich_cols].fillna(0)
    print(f"Edges after {enrich_name} merge: {len(edges):,}")

print(f"\nFinal edge table: {len(edges):,} rows  ×  {len(edges.columns)} columns")
print(f"Columns: {list(edges.columns)}")

In [ ]:
# --- Node feature table ---
# Aggregate edges by src role (outbound) and dst role (inbound) separately,
# then join so every IP has both perspectives per bucket

src_agg = edges.groupby(['bucket_id', 'src_ip']).agg(
    out_degree          = ('dst_ip',            'nunique'),
    out_conn_count      = ('conn_count',         'sum'),
    bytes_sent          = ('orig_bytes',         'sum'),
    bytes_recv_as_src   = ('resp_bytes',         'sum'),
    out_failed_ratio    = ('failed_ratio',       'mean'),
    out_unique_ports    = ('unique_dst_ports',   'sum'),
    out_http_requests   = ('http_requests',      'sum'),
    out_file_count      = ('file_count',         'sum'),
    out_file_bytes      = ('file_bytes',         'sum'),
    out_weird_count     = ('weird_count',        'sum'),
    out_tunnel_count    = ('tunnel_count',       'sum'),
    out_tcp_ratio       = ('tcp_ratio',          'mean'),
).reset_index().rename(columns={'src_ip': 'node_ip'})

dst_agg = edges.groupby(['bucket_id', 'dst_ip']).agg(
    in_degree           = ('src_ip',            'nunique'),
    in_conn_count       = ('conn_count',         'sum'),
    bytes_recv          = ('resp_bytes',         'sum'),
    bytes_sent_as_dst   = ('orig_bytes',         'sum'),
    in_failed_ratio     = ('failed_ratio',       'mean'),
    in_weird_count      = ('weird_count',        'sum'),
    in_tunnel_count     = ('tunnel_count',       'sum'),
).reset_index().rename(columns={'dst_ip': 'node_ip'})

nodes = src_agg.merge(dst_agg, on=['bucket_id', 'node_ip'], how='outer').fillna(0)

# Attach DNS (already keyed by node_ip)
nodes = nodes.merge(dns_node, on=['bucket_id', 'node_ip'], how='left')
dns_cols = ['dns_queries', 'unique_domains', 'nxdomain_count', 'nxdomain_ratio', 'unique_qtypes']
nodes[dns_cols] = nodes[dns_cols].fillna(0)

# Attach IP class label
nodes = nodes.merge(all_ips.rename(columns={'ip': 'node_ip'}), on='node_ip', how='left')
nodes['ip_class'] = nodes['ip_class'].fillna('unknown')

# Derived node-level features
nodes['total_bytes']       = nodes['bytes_sent'] + nodes['bytes_recv']
nodes['degree']            = nodes['out_degree'] + nodes['in_degree']
nodes['total_conn_count']  = nodes['out_conn_count'] + nodes['in_conn_count']
nodes['traffic_asymmetry'] = (
    (nodes['bytes_sent'] - nodes['bytes_recv']) /
    nodes['total_bytes'].replace(0, 1)
)
nodes['bytes_per_conn']    = nodes['total_bytes'] / nodes['total_conn_count'].replace(0, 1)
nodes['weird_rate']        = (
    (nodes['out_weird_count'] + nodes['in_weird_count']) /
    nodes['total_conn_count'].replace(0, 1)
)
nodes['dns_to_conn_ratio'] = nodes['dns_queries'] / nodes['out_conn_count'].replace(0, 1)

print(f"Node table: {len(nodes):,} rows  ×  {len(nodes.columns)} columns")
nodes.head()

In [ ]:
# --- Temporal delta features ---
# Lag-1 difference and 24-bucket rolling z-score per node
# Captures sudden behavioural changes that static snapshots miss

DELTA_COLS = [
    'out_conn_count', 'bytes_sent', 'bytes_recv',
    'out_failed_ratio', 'traffic_asymmetry',
    'dns_queries', 'out_weird_count', 'degree',
]

nodes = nodes.sort_values(['node_ip', 'bucket_id']).reset_index(drop=True)

# Lag-1 delta: how much did this metric change vs the previous hour for this node?
for col in DELTA_COLS:
    nodes[f'delta_{col}'] = nodes.groupby('node_ip')[col].diff().fillna(0)

# Rolling z-score: how many std devs away from the node's recent 24h baseline?
def rolling_zscore(series, window=ROLLING_WINDOW):
    roll  = series.rolling(window, min_periods=3)
    mean  = roll.mean()
    std   = roll.std().replace(0, np.nan)   # avoid div-by-zero on flat signals
    return ((series - mean) / std).fillna(0)

for col in DELTA_COLS:
    nodes[f'zscore_{col}'] = (
        nodes.groupby('node_ip')[col]
             .transform(rolling_zscore, window=ROLLING_WINDOW)
    )

print(f"Node table after temporal features: {len(nodes.columns)} columns")

In [ ]:
# --- Bucket index ---
# Lookup table: bucket_id → wall-clock time, hour-of-day, day-of-week
# Useful for seasonality-aware train/test splits and temporal visualisation

bucket_index = (
    conn_df.groupby('bucket_id')['ts'].min()
    .reset_index()
    .rename(columns={'ts': 'bucket_start_ts'})
)
bucket_index['datetime']    = pd.to_datetime(bucket_index['bucket_start_ts'], unit='s')
bucket_index['hour_of_day'] = bucket_index['datetime'].dt.hour
bucket_index['day_of_week'] = bucket_index['datetime'].dt.dayofweek   # 0=Mon
bucket_index['date']        = bucket_index['datetime'].dt.date
bucket_index['week']        = bucket_index['datetime'].dt.isocalendar().week.astype(int)
bucket_index['conn_count']  = conn_df.groupby('bucket_id')['ts'].count().values
bucket_index['node_count']  = nodes.groupby('bucket_id')['node_ip'].nunique().reindex(
    bucket_index['bucket_id']
).values
bucket_index['edge_count']  = edges.groupby('bucket_id')['src_ip'].count().reindex(
    bucket_index['bucket_id']
).values

print(bucket_index.head(10).to_string(index=False))

In [ ]:
# --- Save outputs ---

edges.to_csv(f'{OUT_DIR}/edges_hourly.csv',   index=False)
nodes.to_csv(f'{OUT_DIR}/nodes_hourly.csv',   index=False)
all_ips.to_csv(f'{OUT_DIR}/ip_registry.csv',  index=False)
bucket_index.to_csv(f'{OUT_DIR}/bucket_index.csv', index=False)

print("Saved to", OUT_DIR)
print(f"  edges_hourly.csv   {len(edges):>8,} rows  ×  {len(edges.columns):>2} cols")
print(f"  nodes_hourly.csv   {len(nodes):>8,} rows  ×  {len(nodes.columns):>2} cols")
print(f"  ip_registry.csv    {len(all_ips):>8,} unique IPs")
print(f"  bucket_index.csv   {len(bucket_index):>8,} hourly buckets")

In [ ]:
# --- Sanity checks ---

print("=== EDGES ===")
print(f"Buckets:            {edges['bucket_id'].nunique()}")
print(f"Unique src IPs:     {edges['src_ip'].nunique()}")
print(f"Unique dst IPs:     {edges['dst_ip'].nunique()}")
print(f"Avg edges/bucket:   {len(edges) / edges['bucket_id'].nunique():.1f}")
missing = edges.isnull().sum()
if missing.any():
    print(f"Missing values:\n{missing[missing > 0]}")
else:
    print("No missing values")

print("\n=== NODES ===")
print(f"Buckets:            {nodes['bucket_id'].nunique()}")
print(f"Unique nodes:       {nodes['node_ip'].nunique()}")
print(f"Avg nodes/bucket:   {len(nodes) / nodes['bucket_id'].nunique():.1f}")
print(f"IP class breakdown (avg per bucket):")
print(
    nodes.groupby(['bucket_id', 'ip_class'])
         .size()
         .groupby(level='ip_class')
         .mean()
         .round(1)
         .to_string()
)

print("\n=== BUCKET INDEX ===")
print(f"Hour-of-day distribution (conn count):")
print(bucket_index.groupby('hour_of_day')['conn_count'].mean().round(0).to_string())